# Mzinga AlphaZero — Hive Training
Self-contained AlphaZero training on Colab GPU.
**W&B**: mzinga-alphazero | **Checkpoints**: Google Drive


In [ ]:
!pip install wandb -q
import torch, numpy
print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}")

In [ ]:
# If opened from Drive, train_colab.py is in the same Drive folder – copy it here
import os, shutil
drive_script = "/content/drive/MyDrive/Colab Notebooks/train_colab.py"
if os.path.exists(drive_script) and not os.path.exists("train_colab.py"):
    shutil.copy(drive_script, "train_colab.py")
    print("Copied train_colab.py from Drive")
# Fallback: upload the zip
if not os.path.exists("train_colab.py"):
    from google.colab import files
    print("Upload mzinga_colab.zip")
    _ = files.upload()
    !unzip -o mzinga_colab.zip
print("train_colab.py ready")
!wc -l train_colab.py

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/mzinga_checkpoints", exist_ok=True)
print("Drive mounted, checkpoint dir ready")

In [ ]:
import wandb
wandb.login(key=input("W&B API key (from https://wandb.ai/authorize): ").strip())

## Start Training
GPU defaults: 512 hidden, 6 blocks, 100 MCTS sims, checkpoints every 10 iters to Drive.

In [ ]:
#@title Training Parameters\nn_iterations = 500  #@param {type:"integer"}\nhidden_dim = 512  #@param {type:"integer"}\nnum_blocks = 6  #@param {type:"integer"}\nnum_sims = 100  #@param {type:"integer"}\ngames_per_iter = 4  #@param {type:"integer"}\nuse_wandb = True  #@param {type:"boolean"}\n\nimport subprocess, sys\ncmd = [\n    sys.executable, "train_colab.py",\n    "--n_iterations", str(n_iterations),\n    "--hidden_dim", str(hidden_dim),\n    "--num_blocks", str(num_blocks),\n    "--num_sims", str(num_sims),\n    "--games_per_iter", str(games_per_iter),\n    "--checkpoint_dir", "/content/drive/MyDrive/mzinga_checkpoints",\n]\nif not use_wandb:\n    cmd.append("--wandb_project")\n    cmd.append("mzinga-alphazero-offline")\nsubprocess.run(cmd)

## Resume from Checkpoint
Run this cell instead of the training cell above if resuming.

In [ ]:
#@title Resume Training\ncheckpoint = "checkpoint_0010.pt"  #@param {type:"string"}\nn_iterations = 500  #@param {type:"integer"}\n\nimport subprocess, sys, os\nckpt_path = os.path.join("/content/drive/MyDrive/mzinga_checkpoints", checkpoint)\nprint(f"Resuming from {ckpt_path}")\nsubprocess.run([\n    sys.executable, "train_colab.py",\n    "--resume", ckpt_path,\n    "--n_iterations", str(n_iterations),\n    "--checkpoint_dir", "/content/drive/MyDrive/mzinga_checkpoints",\n])